In [0]:
dbutils.widgets.text("catalog_name",    "food_inspection")
dbutils.widgets.text("silver_schema",   "silver")
dbutils.widgets.text("gold_schema",     "gold")
dbutils.widgets.text("metadata_schema", "metadata")
 
catalog_name    = dbutils.widgets.get("catalog_name")
silver_schema   = dbutils.widgets.get("silver_schema")
gold_schema     = dbutils.widgets.get("gold_schema")
metadata_schema = dbutils.widgets.get("metadata_schema")
 
spark.sql(f"USE CATALOG {catalog_name}")
print(f"Gold pipeline — {catalog_name}")

In [0]:
from pyspark.sql.functions import (
    col, lit, trim, upper, when, coalesce, concat_ws, md5,
    current_date, current_timestamp, year, month, dayofmonth,
    quarter, dayofweek, date_format, row_number, count as spark_count,
    max as spark_max
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
#load unified silver tables
inspections = spark.table(f"{catalog_name}.{silver_schema}.vw_inspections_unified")
violations  = spark.table(f"{catalog_name}.{silver_schema}.vw_violations_unified")
 
insp_count = inspections.count()
viol_count = violations.count()
print(f"Unified inspections: {insp_count}")
print(f"Unified violations:  {viol_count}")

In [0]:
#dim_date
df_dates = (inspections
    .select(col("inspection_date").alias("full_date"))
    .filter(col("full_date").isNotNull())
    .distinct()
)
 
dim_date = (df_dates
    .withColumn("date_sk", (year("full_date") * 10000 + month("full_date") * 100 + dayofmonth("full_date")).cast("int"))
    .withColumn("day_of_month",  dayofmonth("full_date"))
    .withColumn("day_of_week",   dayofweek("full_date"))
    .withColumn("day_name",      date_format("full_date", "EEEE"))
    .withColumn("month_number",  month("full_date"))
    .withColumn("month_name",    date_format("full_date", "MMMM"))
    .withColumn("quarter",       quarter("full_date"))
    .withColumn("year",          year("full_date"))
    .withColumn("is_weekend",    when(dayofweek("full_date").isin(1, 7), True).otherwise(False))
    .select("date_sk", "full_date", "day_of_month", "day_of_week", "day_name",
            "month_number", "month_name", "quarter", "year", "is_weekend")
)
 
(dim_date.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_date"))
 
dim_date_count = spark.table(f"{catalog_name}.{gold_schema}.dim_date").count()
print(f"dim_date: {dim_date_count} rows")

In [0]:
#dim_violation
dim_viol_src = (violations
    .filter(col("violation_code").isNotNull() & (col("violation_code") != "NO_VIOLATION"))
    .select("violation_code", "violation_description", "source_city")
    .distinct()
)
 
w = Window.orderBy("source_city", "violation_code", "violation_description")
dim_violation = (dim_viol_src
    .withColumn("violation_sk", row_number().over(w).cast("bigint"))
    .select("violation_sk", "violation_code", "violation_description", "source_city")
)
 
(dim_violation.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_violation"))
 
dim_viol_count = spark.table(f"{catalog_name}.{gold_schema}.dim_violation").count()
print(f"dim_violation: {dim_viol_count} rows")

In [0]:
# Verify both cities represented
display(spark.sql(f"""
    SELECT source_city, COUNT(*) AS violation_codes
    FROM {catalog_name}.{gold_schema}.dim_violation
    GROUP BY source_city
"""))

In [0]:
#Dim_Establishment
dim_est_src = (inspections
    .select(
        col("establishment_name"),
        col("aka_name"),
        col("license_number"),
        col("facility_type"),
        col("risk_category"),
        col("address"),
        col("city"),
        col("state"),
        col("zip_code"),
        col("latitude"),
        col("longitude"),
        col("source_city"),
        col("inspection_date")
    )
    .filter(col("establishment_name").isNotNull())
    .distinct()
)
 
# Generate natural key
dim_est_src = dim_est_src.withColumn("establishment_nk",
    md5(concat_ws("||",
        coalesce(upper(trim(col("establishment_name"))), lit("")),
        coalesce(upper(trim(col("address"))), lit("")),
        coalesce(trim(col("zip_code")), lit("")),
        col("source_city")
    ))
)

# Keep the LATEST record per establishment for SCD2 comparison
w_dedup = Window.partitionBy("establishment_nk").orderBy(col("inspection_date").desc())
dim_est_src = (dim_est_src
    .withColumn("_rn", row_number().over(w_dedup))
    .filter(col("_rn") == 1)
    .drop("_rn", "inspection_date")
)
 
est_src_count = dim_est_src.count()
print(f"Distinct establishments from Silver: {est_src_count}")


In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.{gold_schema}.dim_establishment (
        establishment_sk      BIGINT,
        establishment_nk      STRING,
        establishment_name    STRING,
        aka_name              STRING,
        license_number        STRING,
        facility_type         STRING,
        risk_category         STRING,
        address               STRING,
        city                  STRING,
        state                 STRING,
        zip_code              STRING,
        latitude              DOUBLE,
        longitude             DOUBLE,
        source_city           STRING,
        effective_start_date  DATE,
        effective_end_date    DATE,
        is_current            STRING
    )
    USING DELTA
""")
 
print("dim_establishment table ready")

In [0]:
# SCD Type 2 — Initial load or Incremental merge
target_count = spark.table(f"{catalog_name}.{gold_schema}.dim_establishment").count()
 
if target_count == 0:
    # ===== INITIAL LOAD =====
    print("Initial load — inserting all establishments")
    
    w = Window.orderBy("establishment_nk")
    dim_establishment = (dim_est_src
        .withColumn("establishment_sk",     row_number().over(w).cast("bigint"))
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date",   lit("9999-12-31").cast("date"))
        .withColumn("is_current",           lit("Y"))
        .select(
            "establishment_sk", "establishment_nk", "establishment_name",
            "aka_name", "license_number", "facility_type", "risk_category",
            "address", "city", "state", "zip_code", "latitude", "longitude",
            "source_city", "effective_start_date", "effective_end_date", "is_current"
        )
    )
    
    (dim_establishment.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{catalog_name}.{gold_schema}.dim_establishment"))
    
    initial_count = spark.table(f"{catalog_name}.{gold_schema}.dim_establishment").count()
    print(f"Initial load complete: {initial_count} establishments")
 
else:
    # ===== INCREMENTAL SCD2 MERGE =====
    print(f"Incremental merge — target has {target_count} existing records")
    
    dim_est_src.createOrReplaceTempView("incoming_establishments")
    target = DeltaTable.forName(spark, f"{catalog_name}.{gold_schema}.dim_establishment")
    
    # Step 1: Close changed records (set is_current = N, end_date = today)
    target.alias("t").merge(
        dim_est_src.alias("s"),
        "t.establishment_nk = s.establishment_nk AND t.is_current = 'Y'"
    ).whenMatchedUpdate(
        condition="""
            COALESCE(t.establishment_name, '') <> COALESCE(s.establishment_name, '')
            OR COALESCE(t.aka_name, '')        <> COALESCE(s.aka_name, '')
            OR COALESCE(t.facility_type, '')   <> COALESCE(s.facility_type, '')
            OR COALESCE(t.risk_category, '')   <> COALESCE(s.risk_category, '')
            OR COALESCE(t.address, '')         <> COALESCE(s.address, '')
            OR COALESCE(t.zip_code, '')        <> COALESCE(s.zip_code, '')
        """,
        set={
            "is_current":        "'N'",
            "effective_end_date": "current_date()"
        }
    ).execute()
    
    closed_count = spark.sql(f"""
        SELECT COUNT(*) FROM {catalog_name}.{gold_schema}.dim_establishment 
        WHERE is_current = 'N' AND effective_end_date = current_date()
    """).collect()[0][0]
    print(f"Step 1: Closed {closed_count} changed records")
    
    # Step 2: Get max surrogate key
    max_sk = spark.sql(f"""
        SELECT COALESCE(MAX(establishment_sk), 0) AS max_sk 
        FROM {catalog_name}.{gold_schema}.dim_establishment
    """).collect()[0]["max_sk"]
    
    # Step 3: Find records that need new rows (changed + brand new)
    new_records = spark.sql(f"""
        SELECT s.*
        FROM incoming_establishments s
        LEFT JOIN {catalog_name}.{gold_schema}.dim_establishment t
            ON s.establishment_nk = t.establishment_nk AND t.is_current = 'Y'
        WHERE t.establishment_nk IS NULL
    """)
    
    new_count = new_records.count()
    
    if new_count > 0:
        w = Window.orderBy("establishment_nk")
        new_inserts = (new_records
            .withColumn("establishment_sk", (row_number().over(w) + max_sk).cast("bigint"))
            .withColumn("effective_start_date", current_date())
            .withColumn("effective_end_date",   lit("9999-12-31").cast("date"))
            .withColumn("is_current",           lit("Y"))
            .select(
                "establishment_sk", "establishment_nk", "establishment_name",
                "aka_name", "license_number", "facility_type", "risk_category",
                "address", "city", "state", "zip_code", "latitude", "longitude",
                "source_city", "effective_start_date", "effective_end_date", "is_current"
            )
        )
        
        (new_inserts.write.format("delta")
            .mode("append")
            .saveAsTable(f"{catalog_name}.{gold_schema}.dim_establishment"))
        
        print(f"Step 2: Inserted {new_count} new/changed records")
    else:
        print("Step 2: No new or changed records to insert")

In [0]:
#verify dim_eastablishment
print("=== dim_establishment summary ===")
display(spark.sql(f"""
    SELECT 
        source_city,
        COUNT(*) AS total_records,
        SUM(CASE WHEN is_current = 'Y' THEN 1 ELSE 0 END) AS current_records,
        SUM(CASE WHEN is_current = 'N' THEN 1 ELSE 0 END) AS historical_records
    FROM {catalog_name}.{gold_schema}.dim_establishment
    GROUP BY source_city
"""))

In [0]:
# Load dimensions for lookups
dim_est  = (spark.table(f"{catalog_name}.{gold_schema}.dim_establishment")
    .filter(col("is_current") == "Y")
    .select("establishment_sk", "establishment_nk"))
 
dim_dt = spark.table(f"{catalog_name}.{gold_schema}.dim_date").select("date_sk", "full_date")
 
# COMMAND ----------
 
# Generate establishment_nk on inspections (same logic as dim load)
fact_src = inspections.withColumn("establishment_nk",
    md5(concat_ws("||",
        coalesce(upper(trim(col("establishment_name"))), lit("")),
        coalesce(upper(trim(col("address"))), lit("")),
        coalesce(trim(col("zip_code")), lit("")),
        col("source_city")
    ))
)
 
# Generate date_sk
fact_src = fact_src.withColumn("date_sk",
    (year("inspection_date") * 10000 + month("inspection_date") * 100 + dayofmonth("inspection_date")).cast("int")
)

In [0]:
#pre-compute violation count per inspection
violation_counts = (violations
    .filter(col("violation_code") != "NO_VIOLATION")
    .groupBy("inspection_id")
    .agg(spark_count("*").alias("violation_count"))
)
 
# Join violation counts — inspections with no violations get 0
fact_src = fact_src.join(violation_counts, "inspection_id", "left")
fact_src = fact_src.withColumn("violation_count",
    coalesce(col("violation_count"), lit(0)).cast("int")
)
 
print(f"Violation counts computed for {violation_counts.count()} inspections")

In [0]:
#join dimension keys
fact = (fact_src
    .join(dim_est, "establishment_nk", "left")
    .join(dim_dt, fact_src.date_sk == dim_dt.date_sk, "left")
    .drop(dim_dt.date_sk)
)
 
# Check for null FKs
null_est = fact.filter(col("establishment_sk").isNull()).count()
null_dt  = fact.filter(col("date_sk").isNull()).count()
print(f"Null establishment_sk: {null_est}")
print(f"Null date_sk: {null_dt}")

In [0]:
#generate surrogate key and write in fact_inspection
w = Window.orderBy("inspection_id")
fact_inspection = (fact
    .withColumn("inspection_sk", row_number().over(w).cast("bigint"))
    .select(
        "inspection_sk",
        "establishment_sk",
        col("date_sk"),
        col("inspection_id"),
        col("inspection_type"),
        col("inspection_result"),
        col("inspection_score"),
        col("violation_count"),
        col("source_city")
    )
)
 
(fact_inspection.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_inspection"))
 
fact_count = spark.table(f"{catalog_name}.{gold_schema}.fact_inspection").count()
print(f"fact_inspection: {fact_count} rows")


In [0]:
# Verify fact by source city
display(spark.sql(f"""
    SELECT source_city, COUNT(*) AS inspections,
           AVG(inspection_score) AS avg_score,
           AVG(violation_count) AS avg_violations
    FROM {catalog_name}.{gold_schema}.fact_inspection
    GROUP BY source_city
"""))
 

In [0]:
#build brifge with fk lookups
fact_ins  = spark.table(f"{catalog_name}.{gold_schema}.fact_inspection").select("inspection_sk", "inspection_id")
dim_viol  = spark.table(f"{catalog_name}.{gold_schema}.dim_violation").select("violation_sk", "violation_code", "violation_description", col("source_city").alias("v_source"))
 
# Filter out NO_VIOLATION rows — they don't need a bridge entry
bridge_violations = violations.filter(col("violation_code") != "NO_VIOLATION")
 
bridge_src = (bridge_violations.alias("v")
    .join(fact_ins.alias("f"),
        col("v.inspection_id") == col("f.inspection_id"), "inner")
    .join(dim_viol.alias("dv"),
        (col("v.violation_code")        == col("dv.violation_code")) &
        (col("v.violation_description") == col("dv.violation_description")) &
        (col("v.source_city")           == col("dv.v_source")),
        "left")
    .select(
        col("f.inspection_sk"),
        col("dv.violation_sk"),
        col("v.violation_points"),
        col("v.violation_detail"),
        col("v.violation_memo")
    )
    .filter(col("violation_sk").isNotNull())
)

In [0]:
#generate surrogate key and write in bridge_inspection_violation`
w = Window.orderBy("inspection_sk", "violation_sk")
bridge = (bridge_src
    .withColumn("bridge_sk", row_number().over(w).cast("bigint"))
    .select(
        "bridge_sk",
        "inspection_sk",
        "violation_sk",
        "violation_points",
        "violation_detail",
        "violation_memo"
    )
)
 
(bridge.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.bridge_inspection_violation"))
 
bridge_count = spark.table(f"{catalog_name}.{gold_schema}.bridge_inspection_violation").count()
print(f"bridge_inspection_violation: {bridge_count} rows")

In [0]:
# Verify bridge by source city
display(spark.sql(f"""
    SELECT f.source_city, COUNT(*) AS bridge_rows
    FROM {catalog_name}.{gold_schema}.bridge_inspection_violation b
    JOIN {catalog_name}.{gold_schema}.fact_inspection f ON b.inspection_sk = f.inspection_sk
    GROUP BY f.source_city
"""))

In [0]:
#gold zone table counts
print("=" * 60)
print("GOLD LAYER SUMMARY")
print("=" * 60)
 
for table in ["dim_date", "dim_violation", "dim_establishment", "fact_inspection", "bridge_inspection_violation"]:
    cnt = spark.table(f"{catalog_name}.{gold_schema}.{table}").count()
    cols = len(spark.table(f"{catalog_name}.{gold_schema}.{table}").columns)
    print(f"  {table}: {cnt} rows, {cols} columns")

In [0]:
#referrential integrity checks
print("=== REFERENTIAL INTEGRITY ===\n")
 
# Fact → dim_establishment
orphan_est = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog_name}.{gold_schema}.fact_inspection f
    LEFT JOIN {catalog_name}.{gold_schema}.dim_establishment d 
        ON f.establishment_sk = d.establishment_sk
    WHERE d.establishment_sk IS NULL AND f.establishment_sk IS NOT NULL
""").collect()[0][0]
print(f"Fact → dim_establishment orphans: {orphan_est}")
 
# Fact → dim_date
orphan_dt = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog_name}.{gold_schema}.fact_inspection f
    LEFT JOIN {catalog_name}.{gold_schema}.dim_date d ON f.date_sk = d.date_sk
    WHERE d.date_sk IS NULL AND f.date_sk IS NOT NULL
""").collect()[0][0]
print(f"Fact → dim_date orphans: {orphan_dt}")
 
# Bridge → fact_inspection
orphan_bridge_fact = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog_name}.{gold_schema}.bridge_inspection_violation b
    LEFT JOIN {catalog_name}.{gold_schema}.fact_inspection f ON b.inspection_sk = f.inspection_sk
    WHERE f.inspection_sk IS NULL
""").collect()[0][0]
print(f"Bridge → fact_inspection orphans: {orphan_bridge_fact}")
 
# Bridge → dim_violation
orphan_bridge_viol = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog_name}.{gold_schema}.bridge_inspection_violation b
    LEFT JOIN {catalog_name}.{gold_schema}.dim_violation d ON b.violation_sk = d.violation_sk
    WHERE d.violation_sk IS NULL
""").collect()[0][0]
print(f"Bridge → dim_violation orphans: {orphan_bridge_viol}")
 

In [0]:
#scd 2 verification
print("=== SCD2 — dim_establishment ===")
display(spark.sql(f"""
    SELECT 
        source_city,
        COUNT(*) AS total_records,
        SUM(CASE WHEN is_current = 'Y' THEN 1 ELSE 0 END) AS current,
        SUM(CASE WHEN is_current = 'N' THEN 1 ELSE 0 END) AS historical,
        COUNT(DISTINCT establishment_nk) AS unique_establishments
    FROM {catalog_name}.{gold_schema}.dim_establishment
    GROUP BY source_city
"""))

In [0]:
#sample data from each table
print("=== dim_date (sample) ===")
display(spark.table(f"{catalog_name}.{gold_schema}.dim_date").limit(5))
 
# COMMAND ----------
 
print("=== dim_violation (sample) ===")
display(spark.table(f"{catalog_name}.{gold_schema}.dim_violation").limit(10))
 
# COMMAND ----------
 
print("=== dim_establishment (sample) ===")
display(spark.table(f"{catalog_name}.{gold_schema}.dim_establishment").filter(col("is_current") == "Y").limit(10))
 
# COMMAND ----------
 
print("=== fact_inspection (sample) ===")
display(spark.table(f"{catalog_name}.{gold_schema}.fact_inspection").limit(10))
 
# COMMAND ----------
 
print("=== bridge_inspection_violation (sample) ===")
display(spark.table(f"{catalog_name}.{gold_schema}.bridge_inspection_violation").limit(10))